# LPI Phase 3A — iLINCS leave-one-landmark-out control
Control de sensibilidad para la firma LPI congelada. LINCS L1000 sólo solapa 8/20 genes con la firma; este notebook repite el análisis eliminando uno de esos 8 genes cada vez para comprobar que las conexiones no dependen del propio gen perturbado.

**Esto es un análisis de sensibilidad secundario; no redefine el LPI ni el resultado primario.**


In [ ]:
import requests, pandas as pd, json, os, zipfile, time
UP_FULL = ['BCL2L11', 'CDKN1B', 'GADD45A', 'SESN3', 'SOD2', 'CAT']
DOWN_FULL = ['GHR', 'IGF1', 'IGF1R', 'IRS1', 'IRS2', 'PIK3CA', 'PIK3CB', 'PDPK1', 'AKT1', 'AKT2', 'MTOR', 'RPTOR', 'RPS6KB1', 'EIF4EBP1']
COMMON8 = ['CAT', 'IGF1R', 'GADD45A', 'PIK3CA', 'GHR', 'AKT1', 'CDKN1B', 'EIF4EBP1']

TARGET_GENES = ['GHR','AKT1','PIK3CA','PIK3CB','MTOR','RPTOR','PDPK1']
TARGET_COMPOUNDS = ['KU0060648','LY-294002','TORIN-2','Everolimus','TORIN1','Dactolisib','AZD-8055','TGX 221','GSK-2334470']


In [ ]:
def upload_signature(up, down, label):
    lines=['Name_GeneSymbol\tValue_LogDiffExp']
    lines += [f'{g}\t1' for g in up]
    lines += [f'{g}\t-1' for g in down]
    text='\n'.join(lines)+'\n'
    fn=f'LPI_{label}.tsv'
    open(fn,'w').write(text)
    with open(fn,'rb') as fh:
        r=requests.post('https://www.ilincs.org/api/SignatureMeta/upload',files={'file':(fn,fh,'text/tab-separated-values')},timeout=180)
    r.raise_for_status()
    j=r.json()
    sf=j.get('status',{}).get('fileName')
    if isinstance(sf,list): sf=sf[0]
    if not sf: raise RuntimeError(f'No fileName for {label}: {j}')
    return sf, fn

def enrichment(sf, lib):
    r=requests.get('https://www.ilincs.org/api/ilincsR/signatureEnrichment',params={'sigFile':sf,'library':lib,'metadata':'TRUE'},timeout=240)
    r.raise_for_status()
    j=r.json()
    return pd.json_normalize(j.get('enrichment',[])), j

def exact_target_hit(df, gene):
    if df.empty or 'categoryName' not in df.columns: return None
    x=df[df['categoryName'].astype(str).str.upper().eq(gene.upper())]
    return None if x.empty else x.sort_values('FDR').iloc[0].to_dict()

def exact_compound_hit(df, comp):
    if df.empty or 'compound' not in df.columns: return None
    x=df[df['compound'].astype(str).str.lower().eq(comp.lower())]
    return None if x.empty else x.sort_values('FDR').iloc[0].to_dict()


In [ ]:
conditions=[('FULL',None)] + [(f'LOO_{g}',g) for g in COMMON8]
gen_rows=[]; chem_rows=[]; raw_files=[]

for label,drop in conditions:
    up=[g for g in UP_FULL if g!=drop]
    down=[g for g in DOWN_FULL if g!=drop]
    print('\nRunning',label,'UP',len(up),'DOWN',len(down))
    sf, sigfn=upload_signature(up,down,label)
    raw_files.append(sigfn)
    gdf,gj=enrichment(sf,'LIB_6')
    cdf,cj=enrichment(sf,'LIB_5')
    gdf.to_csv(f'{label}_genetic_enrichment.csv',index=False)
    cdf.to_csv(f'{label}_chemical_enrichment.csv',index=False)
    raw_files += [f'{label}_genetic_enrichment.csv',f'{label}_chemical_enrichment.csv']

    for gene in TARGET_GENES + ['CAT','BCL2L11']:
        hit=exact_target_hit(gdf,gene)
        gen_rows.append({
          'condition':label,'dropped_gene':drop or '', 'target_gene':gene,
          'present_in_enrichment':hit is not None,
          'direction': '' if hit is None else hit.get('connDirection',''),
          'zScore': None if hit is None else hit.get('zScore'),
          'pValue': None if hit is None else hit.get('pValue'),
          'FDR': None if hit is None else hit.get('FDR'),
          'nSignature': None if hit is None else hit.get('nSignature')})

    for comp in TARGET_COMPOUNDS:
        hit=exact_compound_hit(cdf,comp)
        chem_rows.append({
          'condition':label,'dropped_gene':drop or '', 'compound':comp,
          'present_in_enrichment':hit is not None,
          'direction': '' if hit is None else hit.get('connDirection',''),
          'zScore': None if hit is None else hit.get('zScore'),
          'pValue': None if hit is None else hit.get('pValue'),
          'FDR': None if hit is None else hit.get('FDR'),
          'nSignature': None if hit is None else hit.get('nSignature')})
    time.sleep(1)

gen_summary=pd.DataFrame(gen_rows)
chem_summary=pd.DataFrame(chem_rows)
gen_summary.to_csv('LPI_iLINCS_LOO_genetic_summary.csv',index=False)
chem_summary.to_csv('LPI_iLINCS_LOO_chemical_summary.csv',index=False)
display(gen_summary.head(20))
display(chem_summary.head(20))


In [ ]:
# Resumen automático de robustez
print('\nGENETIC TARGET ROBUSTNESS')
for gene in ['GHR','AKT1','CAT']:
    x=gen_summary[gen_summary.target_gene.eq(gene)]
    print(gene, 'positive=',sum(x.direction.eq('+')),'negative=',sum(x.direction.eq('-')),
          'FDR<0.05=',sum(pd.to_numeric(x.FDR,errors='coerce')<0.05),'of',len(x))

print('\nCHEMICAL ROBUSTNESS')
for comp in TARGET_COMPOUNDS:
    x=chem_summary[chem_summary.compound.eq(comp)]
    print(comp, 'positive=',sum(x.direction.eq('+')),'negative=',sum(x.direction.eq('-')),
          'FDR<0.05=',sum(pd.to_numeric(x.FDR,errors='coerce')<0.05),'of',len(x))


In [ ]:
outfiles=raw_files+['LPI_iLINCS_LOO_genetic_summary.csv','LPI_iLINCS_LOO_chemical_summary.csv']
with zipfile.ZipFile('LPI_iLINCS_LOO_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for fn in outfiles:
        if os.path.exists(fn): z.write(fn)
print('Created LPI_iLINCS_LOO_results.zip')
try:
    from google.colab import files
    files.download('LPI_iLINCS_LOO_results.zip')
except Exception as e:
    print(e)
